# Locy Use Case: Infrastructure Blast Radius

Compute transitive downstream impact from a failing upstream service.

This notebook uses **schema-first mode** (recommended): labels, edge types, and typed properties are defined before ingest.


## How To Read This Notebook

- Step 1 initializes an isolated local database.
- Step 2 defines schema (the recommended production path).
- Step 3 seeds a minimal graph for this use case.
- Step 4 declares Locy rules and query statements.
- Steps 5-6 evaluate and inspect command/query outputs.
- Step 7 tells you what to look for in the results.


## 1) Setup

Creates a temporary database directory so the example is reproducible and leaves no state behind.


In [ ]:
import os
import shutil
import tempfile
from pprint import pprint

import uni_db

DB_DIR = tempfile.mkdtemp(prefix="uni_locy_")
print("DB_DIR:", DB_DIR)

db = uni_db.Database(DB_DIR)


## 2) Define Schema (Recommended)

Define labels, property types, and edge types before inserting data.


In [ ]:
(
    db.schema()
    .label("Service")
        .property("name", "string")
    .done()
    .edge_type("CALLS", ["Service"], ["Service"])
    .done()
    .apply()
)

print('Schema created')


## 3) Seed Graph Data

Insert only the entities/relationships needed for this scenario so rule behavior stays easy to inspect.


In [ ]:
db.execute("CREATE (:Service {name: 'api'})")
db.execute("CREATE (:Service {name: 'gateway'})")
db.execute("CREATE (:Service {name: 'worker'})")
db.execute("CREATE (:Service {name: 'db'})")
db.execute("CREATE (:Service {name: 'cache'})")
db.execute("MATCH (a:Service {name:'api'}), (g:Service {name:'gateway'}) CREATE (a)-[:CALLS]->(g)")
db.execute("MATCH (g:Service {name:'gateway'}), (w:Service {name:'worker'}) CREATE (g)-[:CALLS]->(w)")
db.execute("MATCH (w:Service {name:'worker'}), (d:Service {name:'db'}) CREATE (w)-[:CALLS]->(d)")
db.execute("MATCH (w:Service {name:'worker'}), (c:Service {name:'cache'}) CREATE (w)-[:CALLS]->(c)")
print('Seeded graph data')


## 4) Locy Program

`CREATE RULE` defines derived relations. `QUERY ... WHERE ... RETURN ...` reads from those relations.


In [ ]:
program = r'''
CREATE RULE impacts AS
MATCH (a:Service)-[:CALLS]->(b:Service)
YIELD KEY a, KEY b

CREATE RULE impacts AS
MATCH (a:Service)-[:CALLS]->(mid:Service)
WHERE mid IS impacts TO b
YIELD KEY a, KEY b

QUERY impacts WHERE a.name = 'api' RETURN b.name AS impacted_service
'''
print(program)


## 5) Evaluate Locy Program

Run the program, then inspect materialization stats (`iterations`, `strata`, and executed queries).


In [ ]:
out = db.locy_evaluate(program)

print("Derived relations:", list(out["derived"].keys()))
stats = out["stats"]
print("Iterations:", stats.total_iterations)
print("Strata:", stats.strata_evaluated)
print("Queries executed:", stats.queries_executed)


## 6) Inspect Command Results

Each command result can contain `rows`; this is the easiest way to verify your rule outputs and query projections.


In [ ]:
print("Derived relation snapshots:")
for rel_name, rel_rows in out["derived"].items():
    print(f"\\n{rel_name}: {len(rel_rows)} row(s)")
    pprint(rel_rows)

if out["command_results"]:
    print("\\nCommand results:")
for i, cmd in enumerate(out["command_results"], start=1):
    print(f"\\nCommand #{i}:", cmd.get("type"))
    rows = cmd.get("rows")
    if rows is not None:
        pprint(rows)
if not out["command_results"]:
    print("\\nNo QUERY/EXPLAIN/ABDUCE command outputs in this program.")


## 7) What To Expect

Use these checks to validate output after evaluation:
- For `api`, impacted services should include `gateway`, `worker`, `db`, and `cache`.
- Rows should represent transitive reachability, not only direct neighbors.
- This pattern is useful for outage simulation and dependency triage.


## 8) Cleanup

Delete the temporary database directory created in setup.


In [ ]:
shutil.rmtree(DB_DIR, ignore_errors=True)
print("Cleaned up", DB_DIR)
